# Current real-session audit
Official JPL arrivals, departures, identities and requests are recorded; alternative-policy delivery/cost are simulated. Run from the project root. Historical synthetic cells follow and are labeled separately.

In [ ]:
from pathlib import Path
import pandas as pd
root = Path.cwd() if Path("results").exists() else Path.cwd().parent
pd.read_csv(root / "results/real_data/coverage.csv")

In [ ]:
scores = pd.read_csv(root / "results/real_data/test_scores.csv")
scores.loc[scores.capacity_fraction.eq(.35), ["period", "policy", "tail", "delivered", "cost_per_kwh", "fallbacks"]]

In [ ]:
pd.read_csv(root / "results/real_data/paired_intervals.csv")

## Historical synthetic audit
The following cells concern the earlier synthetic study; they do not describe the current real-session results.

# EV charging fairness audit

All controller results are synthetic. The real ACN discovery sample supports only a failed coverage gate. This notebook reads saved evidence; it does not tune policies or rerun expensive experiments.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd() if (Path.cwd() / 'results').exists() else Path.cwd().parent
R = ROOT / 'results'
audit = json.loads((R / 'data_audit.json').read_text())
pd.Series(audit)

## Data inclusion

The original audit retains 12 of 26 records from eight users. Exact-time request ordering retains 14 sessions from ten users. Neither audit has a user with five eligible sessions; no empirical fairness estimate is justified.

In [ ]:
assert audit['raw_sessions'] == 26
assert audit['eligible_sessions'] == 12
assert max(map(int, audit['repeat_count_distribution'])) < 5
scores = pd.read_csv(R / 'test_scores.csv')
assert len(scores) == 220
scores.groupby(['capacity_fraction', 'policy'])[['tail','delivered','cost_per_kwh']].mean()

## Locked primary comparison

Negative tail differences favor history-plus-uncertainty. These intervals concern generated populations, not actual drivers or sites. The five-point target was not met.

In [ ]:
lock = json.loads((R / 'protocol_lock.json').read_text())
print('Locked comparator:', lock['comparator'])
pd.read_csv(R / 'tables' / 'paired_intervals.csv').query("policy == 'history'")

## Stronger and simpler alternatives

Simple history sharing has slightly better mean tail service, with an inconclusive paired contrast. The original point-MPC baseline is sensitive to numerical ties; the stronger lexicographic diagnostic is exploratory.

In [ ]:
pd.read_csv(R / 'tables' / 'history_vs_simple_sharing.csv')

In [ ]:
pd.read_csv(R / 'tables' / 'point_lexicographic_diagnostic.csv')[['seed','tail','delivered','fallbacks']]

## Forecast quality and event rarity

Each session contributes one landmark forecast per lead. Event counts are essential: there are only 18 departures at the 60-minute lead across 3,769 eligible landmark sessions.

In [ ]:
f = pd.read_csv(R / 'tables' / 'forecast_scores.csv')
f.groupby(['model','lead_minutes']).agg(events=('events','sum'),sessions=('sessions','sum'),brier=('brier','mean'),logloss=('logloss','mean'))

## Capacity and user-service distributions

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.imshow(plt.imread(ROOT / 'figures' / 'capacity_audit.png'))
ax.axis('off')
plt.show()

In [ ]:
pd.read_csv(R / 'tables' / 'strata.csv').query("policy == 'history' and dimension == 'cohort'")

## Correctness, reproducibility and limits

Numerical feasibility is not grid safety. The clean-environment check is same-machine automated reproduction and does not replace independent human review.

In [ ]:
verification = json.loads((R / 'reproduction' / 'verification.json').read_text())
assert verification['passed']
assert scores.max_violation.max() <= 1e-7
verification

## Post-primary second-pass audit
These diagnostics reuse inspected populations and do not replace the frozen primary results. See `docs/diagnostics.md` for interpretation and the revised data gate.

In [ ]:
from pathlib import Path
import pandas as pd
second_pass = Path("results/second_pass")
if not second_pass.exists():
    second_pass = Path("../results/second_pass")
matched = pd.read_csv(second_pass / "lexicographic_paired_intervals.csv")
display(matched[matched.metric == "tail"])
display(pd.read_csv(second_pass / "forecast_session_balanced.csv").groupby(["model", "lead_minutes"])[["brier", "logloss"]].mean())
